## Classify phrases with ParaDetect (DeBERTa-v3-large + LoRA)

Loads [`srikanthgali/paradetect-deberta-v3-lora`](https://huggingface.co/srikanthgali/paradetect-deberta-v3-lora)
(a LoRA adapter on `microsoft/deberta-v3-large`, ~99% accuracy human-vs-AI
text classifier) and exposes `classify_phrases(phrases, label_name)` so any
list of phrases — e.g. the `fiction_phrases` / `news_phrases` sets extracted
above — can be run through it.

In [ ]:
!pip install -q -U transformers peft accelerate torchao

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
import torch

PARADETECT_REPO = "srikanthgali/paradetect-deberta-v3-lora"

paradetect_tokenizer = AutoTokenizer.from_pretrained(PARADETECT_REPO)
_base_model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-large", num_labels=2
)
paradetect_model = PeftModel.from_pretrained(_base_model, PARADETECT_REPO)

_device = "cuda" if torch.cuda.is_available() else "cpu"
paradetect_model.to(_device)
paradetect_model.eval()
print(f"ParaDetect loaded on {_device}")

In [ ]:
def predict_text_origin(text):
    inputs = paradetect_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(_device)
    with torch.no_grad():
        logits = paradetect_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    return {
        "prediction": "AI" if pred == 1 else "Human",
        "human_probability": probs[0][0].item(),
        "ai_probability": probs[0][1].item(),
    }

def classify_phrases(phrases, label_name=""):
    results = []
    for text in phrases:
        pred = predict_text_origin(text)
        pred["text"] = text
        results.append(pred)
    n_ai = sum(1 for r in results if r["prediction"] == "AI")
    n_human = len(results) - n_ai
    print(f"{label_name}: {len(results)} phrases -> predicted AI={n_ai}  Human={n_human}")
    return results

In [ ]:
fiction_predictions = classify_phrases(fiction_phrases, label_name="Fiction")
news_predictions = classify_phrases(news_phrases, label_name="News")